# Alternative Algorithms Evaluation

Compares Random Forest, XGBoost, Neural Network (ANN), and SVR on the same 80/20
split and 5-fold KFold used in `01_random_forest.ipynb` (seed=42).

The Neural Network (ANN) is the study's own PyTorch model (`02_neural_network.ipynb`),
evaluated under the same protocol as the other three: predictions on the same
44-sample test set, and the 5-fold CV score (over the 172 train+validation
samples) reused directly from `metrics.json` rather than recomputed.

An earlier version of this comparison used a different, much weaker
scikit-learn multi-layer perceptron model (R² ≈ 0.61) here instead, which
caused confusion with the study's actual neural network. It has been removed.

## Split structure
```
Full dataset  (216 samples)
  └─ Test set   (44 samples, 20%)  ← same test set as all other notebooks
  └─ Train set  (172 samples, 80%)
```

In [ ]:
import random
import time
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split, cross_val_score, KFold, learning_curve
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

import sys
sys.path.insert(0, '../src')
import nn_inference

# ── Determinism ──────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cpu')

# Visual configuration
sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12})

with open('../metrics.json') as f:
    metrics = json.load(f)

## Data Loading and Preparation

In [ ]:
df = pd.read_csv('../data/raw/hepg2.csv')

X = df[['% DMSO', 'TREHALOSE']]
y = df['VIABILIDADE']

# Shared 80/20 split (seed=42; identical to random_forest.ipynb)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

# Scaling for SVR (the Neural Network below fits its own scaler on its own 129/43 split)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Shared KFold (same as random_forest.ipynb)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## Training and Cross-Validation (All Models)

In [ ]:
print('--- CROSS-VALIDATION EVALUATION (5 FOLDS) ---')

# 1. Random Forest — load the model saved by 01_random_forest.ipynb
rf_model = joblib.load('../models/random_forest_model.pkl')
y_pred_rf = rf_model.predict(X_test)
cv_rf = cross_val_score(rf_model, X_train, y_train, cv=cv, scoring='r2').mean()
print(f'Random Forest         -> R² Test: {r2_score(y_test, y_pred_rf):.4f} | R² CV: {cv_rf:.4f}')

# 2. Neural Network (ANN) — the study's own PyTorch model. Test-set predictions
# come from the NumPy-only export (nn_inference.py); the 5-fold CV score is
# reused directly from metrics.json, not recomputed here.
y_pred_nn = np.array([nn_inference.predict(d, t) for d, t in X_test.values])
cv_nn = metrics['neural_network']['cv_r2_mean']
print(f'Neural Network (ANN)  -> R² Test: {r2_score(y_test, y_pred_nn):.4f} | R² CV: {cv_nn:.4f}')

# 3. Support Vector Regression — raw scaled features
svr_model = SVR(kernel='rbf', C=100, gamma=0.1)
svr_model.fit(X_train_scaled, y_train)
y_pred_svr = svr_model.predict(X_test_scaled)
cv_svr = cross_val_score(svr_model, X_train_scaled, y_train, cv=cv, scoring='r2').mean()
print(f'SVR                   -> R² Test: {r2_score(y_test, y_pred_svr):.4f} | R² CV: {cv_svr:.4f}')

# 4. XGBoost
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1,
                          random_state=SEED, verbosity=0)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
cv_xgb = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='r2').mean()
print(f'XGBoost               -> R² Test: {r2_score(y_test, y_pred_xgb):.4f} | R² CV: {cv_xgb:.4f}')

## Learning Curves

In [ ]:
def plot_learning_curve(estimator, title, X_data, y_data, ax):
    """Compute and plot learning curve on a subplot."""
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X_data, y_data, cv=cv, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 5), scoring='r2'
    )
    train_mean = np.mean(train_scores, axis=1)
    train_std  = np.std(train_scores,  axis=1)
    test_mean  = np.mean(test_scores,  axis=1)
    test_std   = np.std(test_scores,   axis=1)

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Number of Training Samples')
    ax.set_ylabel('Score (R²)')
    ax.fill_between(train_sizes, train_mean - train_std,
                    train_mean + train_std, alpha=0.1, color='r')
    ax.fill_between(train_sizes, test_mean - test_std,
                    test_mean + test_std,  alpha=0.1, color='g')
    ax.plot(train_sizes, train_mean, 'o-', color='r', label='Training Score')
    ax.plot(train_sizes, test_mean,  'o-', color='g', label='Cross-Validation Score')
    ax.legend(loc='lower right', fontsize=9)


# ------------------------------------------------------------------
# Neural Network (ANN) learning curve: sklearn's learning_curve() cannot be
# used directly (the NN needs its own train/validation split, not CV folds),
# so it is built manually -- 5 increasing fractions of the study's 129-sample
# fit set, 3 seeds per fraction (15 trainings total), evaluated on the fixed
# 43-sample validation set using the winning configuration from metrics.json.
# ------------------------------------------------------------------
class HepatoCryoNN(nn.Module):
    def __init__(self, input_dim=2, use_bn=False, dropout_rate=0.0):
        super().__init__()
        dims = [input_dim, 128, 64, 32]
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if use_bn:
                layers.append(nn.BatchNorm1d(dims[i + 1]))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
        layers.append(nn.Linear(32, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


def train_nn_subset(X_tr, y_tr, X_vl, y_vl, seed, max_epochs=2000, patience=100, weight_decay=1e-4):
    torch.manual_seed(seed)
    np.random.seed(seed)
    X_tr_t = torch.FloatTensor(X_tr)
    y_tr_t = torch.FloatTensor(y_tr).unsqueeze(1)
    X_vl_t = torch.FloatTensor(X_vl)
    y_vl_t = torch.FloatTensor(y_vl).unsqueeze(1)

    batch_size = min(32, len(X_tr))
    gen = torch.Generator()
    gen.manual_seed(seed)
    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size,
                         shuffle=True, drop_last=(len(X_tr) > batch_size), generator=gen)

    model = HepatoCryoNN(input_dim=2, use_bn=False, dropout_rate=0.0)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=30, min_lr=1e-6)

    best_val_loss = float('inf')
    best_state = None
    no_improve = 0
    for epoch in range(max_epochs):
        model.train()
        for X_b, y_b in loader:
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_vl_t), y_vl_t).item()
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break
    model.load_state_dict(best_state)
    model.eval()
    return model


# Rebuild the NN's own 129/43 fit/validation split (identical procedure to 02_neural_network.ipynb)
X_trainval_nn, X_test_nn, y_trainval_nn, y_test_nn = train_test_split(
    X.values, y.values, test_size=0.2, random_state=SEED
)
X_fit_nn, X_val_nn, y_fit_nn, y_val_nn = train_test_split(
    X_trainval_nn, y_trainval_nn, test_size=0.25, random_state=SEED
)
nn_scaler = StandardScaler()
X_fit_nn_scaled = nn_scaler.fit_transform(X_fit_nn)
X_val_nn_scaled = nn_scaler.transform(X_val_nn)

nn_fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
nn_seeds = [0, 1, 42]
nn_sizes = [round(f * len(X_fit_nn_scaled)) for f in nn_fractions]

nn_train_scores = np.zeros((len(nn_sizes), len(nn_seeds)))
nn_val_scores = np.zeros((len(nn_sizes), len(nn_seeds)))

print('--- NEURAL NETWORK LEARNING CURVE (5 sizes x 3 seeds = 15 trainings) ---\n')
t0 = time.time()
for i, (frac, size) in enumerate(zip(nn_fractions, nn_sizes)):
    for j, seed in enumerate(nn_seeds):
        rs = np.random.RandomState(seed)
        idx = rs.permutation(len(X_fit_nn_scaled))[:size]
        X_sub, y_sub = X_fit_nn_scaled[idx], y_fit_nn[idx]

        t1 = time.time()
        m = train_nn_subset(X_sub, y_sub, X_val_nn_scaled, y_val_nn, seed=seed)
        m.eval()
        with torch.no_grad():
            pred_tr = m(torch.FloatTensor(X_sub)).numpy().flatten()
            pred_vl = m(torch.FloatTensor(X_val_nn_scaled)).numpy().flatten()
        r2_tr = r2_score(y_sub, pred_tr)
        r2_vl = r2_score(y_val_nn, pred_vl)
        nn_train_scores[i, j] = r2_tr
        nn_val_scores[i, j] = r2_vl
        print(f'  size={size:3d} (frac={frac:.1f}) seed={seed:3d} -> '
              f'train R²={r2_tr:.4f} | val R²={r2_vl:.4f} ({time.time()-t1:.1f}s) '
              f'total={time.time()-t0:.0f}s')

nn_train_mean = nn_train_scores.mean(axis=1)
nn_train_std = nn_train_scores.std(axis=1)
nn_val_mean = nn_val_scores.mean(axis=1)
nn_val_std = nn_val_scores.std(axis=1)
print(f'\nNN learning curve complete in {time.time()-t0:.0f}s')


fig, ax = plt.subplots(2, 2, figsize=(15, 12))

plot_learning_curve(rf_model,  'Learning Curve: Random Forest', X_train,        y_train, ax[0, 0])
plot_learning_curve(xgb_model, 'Learning Curve: XGBoost',       X_train,        y_train, ax[0, 1])
plot_learning_curve(svr_model, 'Learning Curve: SVR',           X_train_scaled, y_train, ax[1, 0])

ax[1, 1].set_title('Learning Curve: Neural Network (ANN)', fontweight='bold')
ax[1, 1].set_xlabel('Number of Training Samples')
ax[1, 1].set_ylabel('Score (R²)')
ax[1, 1].fill_between(nn_sizes, nn_train_mean - nn_train_std, nn_train_mean + nn_train_std, alpha=0.1, color='r')
ax[1, 1].fill_between(nn_sizes, nn_val_mean - nn_val_std, nn_val_mean + nn_val_std, alpha=0.1, color='g')
ax[1, 1].plot(nn_sizes, nn_train_mean, 'o-', color='r', label='Training Score')
ax[1, 1].plot(nn_sizes, nn_val_mean, 'o-', color='g', label='Validation Score')
ax[1, 1].legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../static/images/model_comparison_learning_curves.png', dpi=300)
plt.show()

## Model Comparison Scatter Plot

In [ ]:
plt.figure(figsize=(10, 8))

# Perfect prediction line
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2,
         label='Perfect Prediction (In Vitro = In Silico)')

# Tree-based models (robust)
plt.scatter(y_test, y_pred_rf,  alpha=0.7, color='#2ca02c',
            label=f'Random Forest (CV={cv_rf:.2f})', s=70)
plt.scatter(y_test, y_pred_xgb, alpha=0.6, color='#1f77b4',
            label=f'XGBoost (CV={cv_xgb:.2f})', s=70, marker='^')

# Neural Network (ANN) and SVR
plt.scatter(y_test, y_pred_nn, alpha=0.7, color='#d62728',
            label=f'Neural Network (ANN) (CV={cv_nn:.2f})', s=70, marker='X')
plt.scatter(y_test, y_pred_svr, alpha=0.7, color='#ff7f0e',
            label=f'SVR (CV={cv_svr:.2f})', s=70, marker='s')

plt.xlabel('Observed in vitro Viability (%)', fontweight='bold')
plt.ylabel('Predicted in silico Viability (%)', fontweight='bold')
plt.title('Predictive Performance: Model Comparison', fontsize=15, fontweight='bold')
plt.legend()
plt.tight_layout()

plt.savefig('../static/images/model_comparison_scatter.png', dpi=300)
plt.show()